# Learning to ski from NYC and Philly

Day 21 · OpenStreetMap. Every ski area within three hours of New York or Philadelphia, ranked by beginner terrain,
with each mountain's real terrain from USGS 3DEP processed in ArcGIS Pro.

How the page is made, cell by cell:

1. **Settings**: names, limits and the few numbers worth changing.
2. **Load data**: the processed files from `fetch_week1.py day21` and the terrain index from `day21_terrain_arcgis.py`.
3. **Prepare layers**: the ranked list, trail and lift GeoJSON, the overview relief picture, the coarse elevation grid.
4. **Build page**: pour the data into `scripts/day21_page_template.html`.
5. **Export**: `publish.save` writes `output/day21_openstreetmap-ski.html`, a thumbnail, a caption stub and the gallery card.

The page loads each mountain's `day21_terrain_<slug>_shade.jpg`, `_dem.bin` and `_contours.json` from `data/processed/`,
reached from the page as `../data/processed/`, the first time that mountain is clicked, so the page itself stays light. Run it with `python scripts/fetch_week1.py day21` and
`propy.bat scripts/day21_terrain_arcgis.py` done first.

## 1 · Settings

In [1]:
DAY      = 21
PROMPT   = "OpenStreetMap"
SLUG     = "openstreetmap-ski"
NOTEBOOK = "day21_openstreetmap_ski.ipynb"
TITLE    = "Learning to ski from NYC and Philly"
TEMPLATE = "day21_page_template.html"     # next to this notebook in scripts/

MAX_MINUTES  = 180      # the day-trip rule: closest city within three hours, no traffic
TOP_NAMES    = 7        # how many circles carry their name on the overview
RELIEF_PX    = 1500     # width of the overview relief picture
COARSE_PX    = 900      # width of the coarse elevation grid the page builds its 3D tiles from
EXAGGERATION = 1.25     # vertical exaggeration when zoomed in; the page says so

# Resort names as OpenSkiMap has them are long. These pairs shorten them for labels, first match wins.
SHORT_NAMES = [("Bear Creek Mountain Resort Ski Slopes", "Bear Creek"), ("Spring Mountain Ski Area", "Spring Mountain"),
               ("Skytop\u2019s Ski Hill", "Skytop"), ("Blue Mountain Resort", "Blue Mountain"),
               (" Ski Resort Area", ""), (" Ski Resort", ""), (" Mountain Resort", ""), (" Mountain Ski Area", ""),
               (" Ski Area", ""), (" Resort", ""), (" Mountain Club", ""), (" Mountain Ski Center", ""),
               (" Ski Slopes", ""), (" Ski Hill", "")]

CITIES = [["New York", -73.9855, 40.758, "r"], ["Philadelphia", -75.1636, 39.9522, "l"]]   # name, lon, lat, label side

# OpenSkiMap's website tag for a few resorts is not a web address, or points to a page that has gone. These are the
# resorts' current official addresses, each opened and checked on 2026-11-05. Keyed by the ski area's full name.
WEBSITE_FIXES = {"Montage Mountain Ski Area": "https://www.montagemountainresorts.com/",
                 "Roundtop Mountain Resort": "https://www.skiroundtop.com/",
                 "Eagle Rock Ski Resort": "https://ddresorts.com/eagle-rock/",
                 "Victor Constant Ski Area": "https://westpoint.armymwr.com/programs/victor-constant-ski-area"}

## 2 · Load data

In [2]:
import base64, io, json, sys
from pathlib import Path

import numpy as np
import geopandas as gpd
from PIL import Image, ImageDraw
from scipy import ndimage

SCRIPTS = Path.cwd() if (Path.cwd() / "mapkit").exists() else Path.cwd() / "scripts"
sys.path.insert(0, str(SCRIPTS))
from mapkit.paths import PROCESSED, WGS84
from mapkit import publish, style

runs   = gpd.read_file(PROCESSED / "ski_runs.geojson")
lifts  = gpd.read_file(PROCESSED / "ski_lifts.geojson")
areas  = gpd.read_file(PROCESSED / "ski_areas.geojson")
states = gpd.read_file(PROCESSED / "ski_states.geojson")
relief = np.load(PROCESSED / "ski_relief.npz")
index  = json.loads((PROCESSED / "ski_terrain_index.json").read_text(encoding="utf-8"))

# Only the terrain keys are read from the index. Water and tree keys, if present, are ignored.
HIRES = {v["area_id"]: dict(slug=v["slug"], pm=v["bounds_3857"], nx=v["dem_size"][0], ny=v["dem_size"][1]) for v in index.values()}
print(f"{len(runs)} runs, {len(lifts)} lifts, {len(areas)} ski areas, {len(states)} states, relief {relief['dem_dm'].shape}, {len(HIRES)} mountains with terrain")

C:\Users\gauta\.conda\envs\geospatial\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


2039 runs, 371 lifts, 64 ski areas, 11 states, relief (1407, 1800), 39 mountains with terrain


## 3 · Prepare layers

### 3a · The ranked list

Ski areas within the day-trip rule, sorted by beginner km. Each row carries what the list and the right panel show,
the box around its trails, and the camera bearing for the fly-in: the direction from the lowest trail points to the
highest, read off the mountain's own ArcGIS Pro DEM, so the camera looks uphill.

In [3]:
R = 6378137.0
def merc(lon, lat):
    return np.radians(lon) * R, np.log(np.tan(np.pi / 4 + np.radians(lat) / 2)) * R
def unmerc(x, y):
    return float(np.degrees(x / R)), float(np.degrees(2 * np.arctan(np.exp(y / R)) - np.pi / 2))

def short(name):
    for old, new in SHORT_NAMES:
        if old in name:
            return name.replace(old, new)
    return name

def mountain_dem(area_id):
    """One mountain's ArcGIS Pro DEM as a float array in meters, north row first, plus its Web Mercator box."""
    h = HIRES[area_id]
    z = np.fromfile(PROCESSED / f"day21_terrain_{h['slug']}_dem.bin", dtype="<i2").reshape(h["ny"], h["nx"]) / 10.0
    return z, h["pm"]

def dem_at(z, pm, lons, lats):
    """Bilinear elevation at lon/lat points from a mountain's DEM."""
    x, y = merc(np.asarray(lons), np.asarray(lats))
    x0, y0, x1, y1 = pm
    col = (x - x0) / (x1 - x0) * z.shape[1] - 0.5
    row = (y1 - y) / (y1 - y0) * z.shape[0] - 0.5
    return ndimage.map_coordinates(z, [row, col], order=1, mode="nearest")

def uphill_bearing(area_id, area_runs):
    """Compass bearing from the lowest 15% of trail points to the highest 15%, in degrees clockwise from north."""
    pts = np.array([c[:2] for g in area_runs.geometry for part in getattr(g, "geoms", [g])
                    for c in (part.exterior.coords if part.geom_type == "Polygon" else part.coords)])
    z, pm = mountain_dem(area_id)
    e = dem_at(z, pm, pts[:, 0], pts[:, 1])
    lo, hi = pts[e <= np.percentile(e, 15)], pts[e >= np.percentile(e, 85)]
    dlon = (hi[:, 0].mean() - lo[:, 0].mean()) * np.cos(np.radians(pts[:, 1].mean()))
    dlat = hi[:, 1].mean() - lo[:, 1].mean()
    return round(float(np.degrees(np.arctan2(dlon, dlat))), 1)

ranked = areas[areas.closest_min <= MAX_MINUTES].sort_values("beginner_km", ascending=False).reset_index(drop=True)
ranked = ranked[ranked.area_id.isin(HIRES)]          # every listed mountain has terrain; a missing one would be a bug upstream
rows = []
for i, a in enumerate(ranked.itertuples(), 1):
    g = runs[runs.area_id == a.area_id]
    minutes = lambda v: None if v is None or np.isnan(v) else int(v)
    rows.append(dict(rank=i, id=a.area_id, name=short(a.ski_area), lon=round(a.geometry.x, 5), lat=round(a.geometry.y, 5),
                     beg=a.beginner_km, bunny=a.bunny_km, green=a.green_km, total=a.total_km,
                     carpets=int(a.magic_carpets), lifts=int(a.lifts), nyc=minutes(a.nyc_min), phl=minutes(a.philly_min),
                     site=WEBSITE_FIXES.get(a.ski_area, a.website if isinstance(a.website, str) else None),
                     bounds=[round(float(v), 5) for v in g.total_bounds], bearing=uphill_bearing(a.area_id, g)))
print(len(rows), "ski areas within", MAX_MINUTES, "minutes")
for r in rows[:TOP_NAMES]:
    print(f"  {r['rank']:2d} {r['name']:<28} {r['beg']:5.1f} beginner km of {r['total']:5.1f}   NYC {r['nyc']} min, Philly {r['phl']} min, camera {r['bearing']}°")

39 ski areas within 180 minutes
   1 Jiminy Peak                   11.2 beginner km of  18.7   NYC 180 min, Philly 282 min, camera 147.1°
   2 Mountain Creek                 8.7 beginner km of  16.3   NYC 95 min, Philly 164 min, camera 149.4°
   3 Windham                        8.7 beginner km of  29.6   NYC 162 min, Philly 245 min, camera -166.5°
   4 Belleayre                      8.1 beginner km of  42.1   NYC 156 min, Philly 239 min, camera -133.9°
   5 Camelback                      7.5 beginner km of  16.3   NYC 108 min, Philly 123 min, camera -154.4°
   6 Blue Mountain                  7.1 beginner km of  16.8   NYC 131 min, Philly 110 min, camera 172.3°
   7 Catamount                      5.8 beginner km of  18.0   NYC 150 min, Philly 251 min, camera -145.1°


### 3b · Trails and lifts

Each trail gets a class the page styles: `bunny` (novice), `green` (easy) or `rest`. Its length is measured in UTM.
A slope drawn as a shape counts half its outline as its length, keeps a soft fill, and gets an outline only if no
trail line already runs inside it, so nothing is drawn twice.

In [4]:
keep = {r["id"] for r in rows}
r = runs[runs.area_id.isin(keep)].copy()
u = r.to_crs(32618)
is_shape = u.geom_type.isin(["Polygon", "MultiPolygon"])
r["km"] = (np.where(is_shape, u.length / 2, u.length) / 1000).round(2)
r["tid"] = range(len(r))
doubled = gpd.sjoin(u[is_shape][["geometry"]], gpd.GeoDataFrame(geometry=u[~is_shape].representative_point(), crs=u.crs),
                    predicate="contains").index.unique()
r["draw"] = (~is_shape | ~u.index.isin(doubled)).astype(int)
r["fill"] = is_shape.astype(int)
r["cls"] = r.difficulty.map(lambda d: "bunny" if d == "novice" else "green" if d == "easy" else "rest")
LABEL = {"novice": "Bunny hill", "easy": "Green trail", "intermediate": "Blue trail", "advanced": "Black diamond",
         "expert": "Double black", "freeride": "Freeride", "extreme": "Extreme", "unrated": "Unrated"}
r["label"] = r.difficulty.map(LABEL).fillna("Unrated")

l = lifts[lifts.area_id.isin(keep)].copy()
l["cls"] = np.where(l.lift_type == "magic_carpet", "carpet", "lift")

def geojson(gdf, cols):
    """Only the columns the page needs. Coordinates were already rounded to five decimals when fetch_week1.py saved them."""
    return json.loads(gdf[cols + ["geometry"]].to_json(drop_id=True))

runs_j = geojson(r, ["tid", "name", "cls", "label", "km", "area_id", "draw", "fill"])
lifts_j = geojson(l, ["cls", "area_id"])
land = json.loads(gpd.GeoDataFrame(geometry=states.geometry.simplify(0.002), crs=states.crs).to_json(drop_id=True))
print(f"{len(r)} trails ({int(is_shape.sum())} drawn as shapes, {int((r.draw == 0).sum())} of those with a line inside), {len(l)} lifts, {int((l.cls == 'carpet').sum())} magic carpets")

1164 trails (365 drawn as shapes, 284 of those with a line inside), 245 lifts, 80 magic carpets

### 3c · The overview relief

`ski_relief.npz` is the USGS grid for the whole search box, in decimeters. A few cells are specks, far below sea
level or far above any mountain here, so each one is replaced with the median of its neighbours. The hillshade keeps
only shadows, as grey with alpha, so the land stays white. The land mask hides the sea and the edges fade over 70 px
so the picture's boundary never shows.

In [5]:
z = relief["dem_dm"] / 10.0
bb = [float(v) for v in relief["bbox_3857"]]
H, W = z.shape
bad = (z < -20) | (z > 1300)
z[bad] = ndimage.median_filter(z, 5)[bad]
print(f"{int(bad.sum())} specks replaced; elevation now {z.min():.0f} to {z.max():.0f} m")

cell = (bb[2] - bb[0]) / W
gy, gx = np.gradient(ndimage.gaussian_filter(z, 0.6) * 8, cell)          # the 8 makes low hills show at this scale
az, alt = np.radians(315), np.radians(42)
slope = np.pi / 2 - np.arctan(np.hypot(gx, gy))
aspect = np.arctan2(-gx, gy)
shade = np.sin(alt) * np.sin(slope) + np.cos(alt) * np.cos(slope) * np.cos(az - aspect)
shadow = np.clip(-(shade - np.sin(alt)) / 0.45, 0, 1) ** 0.8               # 0 on flat ground, 1 in deep shadow

mask = Image.new("L", (W, H), 0)
draw = ImageDraw.Draw(mask)
def to_px(coords):
    c = np.asarray(coords)
    x, y = merc(c[:, 0], c[:, 1])
    return list(zip((x - bb[0]) / (bb[2] - bb[0]) * W, (bb[3] - y) / (bb[3] - bb[1]) * H))
for g in states.geometry:
    for part in getattr(g, "geoms", [g]):
        draw.polygon(to_px(part.exterior.coords), fill=255)
        for hole in part.interiors:
            draw.polygon(to_px(hole.coords), fill=0)
land_mask = np.array(mask) > 0
yy, xx = np.mgrid[0:H, 0:W]
edge = np.clip(np.minimum.reduce([xx, yy, W - 1 - xx, H - 1 - yy]) / 70, 0, 1)

alpha = (shadow * 0.62 * 255 * land_mask * edge).astype("uint8")
picture = Image.fromarray(np.dstack([np.full((H, W), 92, "uint8"), alpha]), "LA").resize((RELIEF_PX, round(RELIEF_PX * H / W)), Image.LANCZOS)
buf = io.BytesIO(); picture.save(buf, "PNG", optimize=True)
relief_uri = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()
nw, se = unmerc(bb[0], bb[3]), unmerc(bb[2], bb[1])
corners = [[nw[0], nw[1]], [se[0], nw[1]], [se[0], se[1]], [nw[0], se[1]]]
print(f"relief picture {picture.size[0]} x {picture.size[1]} px, {len(relief_uri) / 1e6:.2f} MB embedded")

1185 specks replaced; elevation now -20 to 1226 m


relief picture 1500 x 1172 px, 2.73 MB embedded


### 3d · The coarse elevation grid

The page builds its own 3D terrain tiles from this grid, then blends in each mountain's sharp ArcGIS Pro grid once
loaded. It is the same relief grid, lightly smoothed and shrunk, stored as little-endian 16-bit decimeters.

In [6]:
ch = round(COARSE_PX * H / W)
coarse = ndimage.zoom(ndimage.gaussian_filter(z, 1), (ch / H, COARSE_PX / W), order=1)[:ch, :COARSE_PX]
coarse_b64 = base64.b64encode(np.round(coarse * 10).astype("<i2").tobytes()).decode()
print(f"coarse grid {coarse.shape[1]} x {coarse.shape[0]} cells, about {cell * W / coarse.shape[1] / 1000:.2f} km per cell, {len(coarse_b64) / 1e6:.2f} MB embedded")

coarse grid 900 x 704 cells, about 0.78 km per cell, 1.69 MB embedded


## 4 · Build page

In [7]:
DATA = dict(rows=rows, runs=runs_j, lifts=lifts_j, land=land, relief=relief_uri, corners=corners,
            dem=dict(w=int(coarse.shape[1]), h=int(coarse.shape[0]), bb=bb, coarse=coarse_b64),
            hires={k: v for k, v in HIRES.items() if k in keep}, cities=CITIES, top_names=TOP_NAMES, exaggeration=EXAGGERATION)
template = (SCRIPTS / TEMPLATE).read_text(encoding="utf-8")
assert '"__DATA__"' in template, "the template must contain the \"__DATA__\" placeholder"
html = template.replace('"__DATA__"', json.dumps(DATA, separators=(",", ":"), ensure_ascii=False))
print(f"page {len(html.encode('utf-8')) / 1e6:.2f} MB, {len(rows)} ski areas, {len(runs_j['features'])} trails")

page 5.38 MB, 39 ski areas, 1164 trails


## 5 · Export

In [8]:
CREDITS = "\n".join([
    "- Trails, lifts and ski areas: © OpenStreetMap contributors via OpenSkiMap (https://openskimap.org), ODbL",
    "- Terrain: USGS 3DEP elevation (https://www.usgs.gov/3d-elevation-program), hillshade and contours made in ArcGIS Pro",
    "- Drive times: Mapbox Matrix API, no traffic",
    "- " + style.CREDITS["census"]["full"],
])
publish.save(html, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle="", series="city",
             tool="Python + MapLibre + ArcGIS Pro", notebook=NOTEBOOK, credits=CREDITS)

map        output/day21_openstreetmap-ski.html
thumbnail  png_thumbnail/day21_openstreetmap-ski.png
caption    captions/day21_openstreetmap-ski.md
live link  https://gautamik01.github.io/30-day-map-challenge-2026/output/day21_openstreetmap-ski.html  (after you push)
